# ML-10 — Content Action Playbook

This notebook turns the validated refresh/opportunity scoring work into a practical, human-reviewed content action queue.

**Lane:** Refresh / Content Opportunity Scoring

The output is decision support: it identifies pages that look worth reviewing first. It does **not** prove that a refresh will cause better performance.

The queue is regenerated from the anonymized starter data. Generated CSVs stay in `work/outputs/` and are intentionally excluded from Git.


## 1. Ranked actions + reason codes

The ranking combines three observable signals used in the earlier baseline:

* CTR: lower CTR can indicate an opportunity when a page already receives search impressions.
* Average position: weaker positions can indicate an opportunity for review.
* 90-day impressions: higher visibility makes a potential improvement more valuable to inspect.

The resulting score is a prioritization score, not a probability of improvement.

Each queue row receives a reason code and a plain-language recommended action.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/Roselyn-Koech/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
local_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in local_candidates if p.exists()), None)
df = pd.read_csv(DATA_PATH) if DATA_PATH else pd.read_csv(DATA_URL)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates("content_id").reset_index(drop=True)

numeric_cols = ["ctr", "avg_position", "impressions_90d", "content_age_days", "days_since_last_update"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["position_has_data"] = df["avg_position"].gt(0)
df["position_clean"] = df["avg_position"].where(df["position_has_data"])

print(f"Eligible rows: {len(df):,}")
print(f"Rows with usable average position: {df['position_has_data'].sum():,}")


In [ ]:
def minmax(series):
    lo = series.min()
    hi = series.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.0, index=series.index)
    return (series - lo) / (hi - lo)

ctr_opportunity = 1 - minmax(df["ctr"])
position_opportunity = pd.Series(0.0, index=df.index)
mask = df["position_has_data"]
position_opportunity.loc[mask] = minmax(df.loc[mask, "position_clean"])
impression_visibility = minmax(np.log1p(df["impressions_90d"]))

df["baseline_action_score"] = (
    0.40 * ctr_opportunity
    + 0.35 * position_opportunity
    + 0.25 * impression_visibility
)
cutoff = df["baseline_action_score"].quantile(0.80)

def reason_code(row):
    low_ctr = row["ctr"] <= df["ctr"].quantile(0.25)
    weak_position = row["position_has_data"] and row["avg_position"] >= df.loc[df["position_has_data"], "avg_position"].quantile(0.75)
    stale = row["days_since_last_update"] >= df["days_since_last_update"].quantile(0.75)
    if low_ctr and weak_position:
        return "REFRESH_LOW_CTR_WEAK_POSITION"
    if stale and low_ctr:
        return "REFRESH_STALE_LOW_CTR"
    if weak_position:
        return "REVIEW_WEAK_POSITION"
    if low_ctr:
        return "REVIEW_LOW_CTR"
    if stale:
        return "REVIEW_FRESHNESS"
    return "REVIEW_HIGH_OPPORTUNITY"

def action_for_reason(reason):
    mapping = {
        "REFRESH_LOW_CTR_WEAK_POSITION": "Review title, snippet, intent match and on-page usefulness",
        "REFRESH_STALE_LOW_CTR": "Review freshness, search intent and content coverage",
        "REVIEW_WEAK_POSITION": "Review relevance, structure and topical coverage before changing content",
        "REVIEW_LOW_CTR": "Review SERP presentation and intent alignment",
        "REVIEW_FRESHNESS": "Check whether information or examples need updating",
        "REVIEW_HIGH_OPPORTUNITY": "Human review before selecting a refresh action",
    }
    return mapping[reason]

df["reason_code"] = df.apply(reason_code, axis=1)
df["recommended_action"] = df["reason_code"].map(action_for_reason)
df["priority"] = np.where(df["baseline_action_score"] >= cutoff, "HIGH", "STANDARD")

queue_cols = [
    "content_id", "baseline_action_score", "priority",
    "reason_code", "recommended_action",
    "content_type", "main_intent", "freshness_tier",
    "ctr", "avg_position", "impressions_90d",
    "content_age_days", "days_since_last_update"
]
action_queue = (df[queue_cols].sort_values(["priority", "baseline_action_score"], ascending=[True, False]).reset_index(drop=True))
action_queue["rank"] = np.arange(1, len(action_queue) + 1)
print("Queue rows:", len(action_queue))
display(action_queue.head(20))


### Archetype → action mapping

The score identifies **where to look first**. The archetype determines what a reviewer should inspect.

| Archetype | Review action |
|---|---|
| Low CTR + weaker position | Inspect search intent, title/snippet alignment, and content usefulness |
| Stale + low CTR | Check whether facts, examples, or coverage are outdated |
| Weak position with visibility | Review topical coverage and relevance before changing copy |
| Low CTR with usable position | Inspect SERP presentation and intent alignment |
| Strong visibility / low immediate risk | Protect and monitor rather than making unnecessary changes |


## 2. Intended use and limits

**Intended users:** content strategists, SEO analysts, editors, and reviewers.

**Intended use:** use the queue to decide which pages deserve human attention first during a content refresh cycle.

The score is a prioritization mechanism based on the observed data. It is not a guarantee that changing a page will improve CTR, rankings, traffic, or conversions.

The analysis uses a cross-sectional snapshot of trailing metrics. Therefore, the results should be treated as decision support rather than causal evidence.

The queue should be regenerated when the underlying data changes materially rather than treated as a permanent list.


In [ ]:
summary = pd.DataFrame({
    "metric": ["eligible_pages", "high_priority_pages", "high_priority_share", "median_action_score", "median_ctr", "median_impressions_90d"],
    "value": [len(df), int((df["priority"] == "HIGH").sum()), round((df["priority"] == "HIGH").mean(), 3), round(df["baseline_action_score"].median(), 3), round(df["ctr"].median(), 3), round(df["impressions_90d"].median(), 1)]
})
display(summary)


## 3. Human review + the no-go list

Every high-priority recommendation requires a human review before action.

### Human review checklist

1. Confirm the page's actual search intent.
2. Read the current page before deciding what to change.
3. Check whether the observed signal is supported by enough data.
4. Check current SERP context and competing results.
5. Confirm the recommendation is relevant to the business and page purpose.
6. Record the decision: refresh, expand, protect, monitor, or no action.
7. Re-measure after the agreed review period.

### No-go list

The system should **not** automatically:

* publish or rewrite content;
* delete or prune pages;
* create redirects;
* change canonical tags or indexation;
* change internal links at scale;
* make claims about Google ranking algorithms;
* declare that a refresh will improve performance;
* make business-critical decisions without human review.


## 4. Monitoring / retrain triggers

The playbook is useful only while the underlying data and relationships remain reasonably stable.

### Monitor

* distribution of the action score;
* share of pages receiving each reason code;
* missingness in key fields;
* CTR and position distributions;
* freshness distribution;
* reviewer acceptance or rejection of recommendations.

### Refresh the scoring logic when

* the score distribution shifts materially;
* the share of missing position data changes materially;
* a new content type or measurement definition is introduced;
* the business changes its definition of a useful refresh opportunity;
* reviewer feedback shows that reason codes are repeatedly unhelpful.

### Retrain / revalidate a learned model when

* new labelled outcome data becomes available;
* out-of-sample ranking performance falls below the previously validated level;
* feature definitions or data collection change;
* the time period or client population changes enough to affect comparability.

A monitoring trigger is a reason to inspect and revalidate, not an automatic deployment trigger.


In [ ]:
monitoring = pd.DataFrame({
    "check": ["Missing CTR", "Missing impressions", "Missing position", "High-priority share", "Reason-code coverage"],
    "value": [round(df["ctr"].isna().mean(), 3), round(df["impressions_90d"].isna().mean(), 3), round((~df["position_has_data"]).mean(), 3), round((df["priority"] == "HIGH").mean(), 3), round(action_queue["reason_code"].notna().mean(), 3)]
})
display(monitoring)


## 5. Decay / refresh insight

The earlier analysis treats freshness as a useful review signal, not as proof of causation.

Here we compare the observed declining-label rate across freshness tiers. A difference between tiers is **descriptive**: it tells us where review may be useful, not what a refresh will cause.


In [ ]:
if "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
    freshness_summary = (df.groupby("freshness_tier", dropna=False).agg(pages=("content_id", "size"), declining_rate=("is_declining_label", "mean"), median_ctr=("ctr", "median"), median_position=("position_clean", "median")).sort_values("declining_rate", ascending=False).reset_index())
    display(freshness_summary)
else:
    freshness_summary = pd.DataFrame()
    print("Freshness comparison unavailable because the target column is not present.")


In [ ]:
fig_dir = Path("work/figures")
out_dir = Path("work/outputs")
fig_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

if not freshness_summary.empty:
    plot_df = freshness_summary.dropna(subset=["declining_rate"]).copy()
    plt.figure(figsize=(9, 5))
    plt.bar(plot_df["freshness_tier"].astype(str), plot_df["declining_rate"])
    plt.ylabel("Observed declining rate")
    plt.xlabel("Freshness tier")
    plt.title("Observed declining rate by freshness tier")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(fig_dir / "freshness_decline_rate.png", dpi=160)
    plt.show()


## 6. Cost / value thinking

The queue is designed for limited review capacity.

A practical operating rule is:

**Start with the highest-ranked pages that combine meaningful visibility with a clear review signal.**

This avoids spending equal effort on every page. A page with little visibility may still matter strategically, but its expected near-term value is harder to infer from this dataset alone.

The score therefore helps allocate **review attention**, not automatically allocate publishing resources.


In [ ]:
reason_counts = (action_queue["reason_code"].value_counts().rename_axis("reason_code").reset_index(name="pages"))
display(reason_counts)


## 7. Exports for the paper

The ranked queue is exported to `work/outputs/action_queue.csv`.

The reusable figure is saved to `work/figures/freshness_decline_rate.png`.

The CSV is intentionally not committed because the repository leak guard excludes `work/**/*.csv`. The notebook is the reproducible source for regenerating it.


In [ ]:
queue_path = out_dir / "action_queue.csv"
action_queue.to_csv(queue_path, index=False)
summary_path = out_dir / "action_playbook_summary.json"
summary_payload = {"eligible_pages": int(len(df)), "high_priority_pages": int((df["priority"] == "HIGH").sum()), "high_priority_share": float((df["priority"] == "HIGH").mean()), "median_action_score": float(df["baseline_action_score"].median()), "reason_codes": reason_counts.to_dict(orient="records")}
import json
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)
print(f"Saved queue: {queue_path}")
print(f"Saved summary: {summary_path}")


## 8. Self-check

* [x] Ranked actions and reason codes are generated.
* [x] Intended use and limits are stated.
* [x] Human review rules and no-go cases are explicit.
* [x] Monitoring and retrain triggers are defined.
* [x] Cost/value thinking is included.
* [x] Decay/freshness is described as an observed pattern, not a causal claim.
* [x] Queue export is generated under `work/outputs/`.
* [x] Reusable figure is generated under `work/figures/`.
* [x] The notebook uses careful decision-support language.
* [x] No client names, private queries, or client URLs are included.
* [ ] Run **Runtime → Run all** in Colab and confirm every cell completes successfully.
* [ ] Commit the executed notebook and reusable figure to the repository.
